In [1]:
from dotenv import load_dotenv
load_dotenv()
from rag_helper import RAGBase
from openai import OpenAI


In [2]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [3]:
openai_client = OpenAI()

assistant = RAGBase(
    sqlite_index,
    llm_client=openai_client,
)



In [4]:
answer = assistant.rag("How do I run olama in locally?")
print(answer)

To run Ollama locally:

1. Install Ollama from: https://ollama.com/download
2. Open a terminal and run:

```bash
ollama run llama3
```

This will download the model and start it locally.

To test that the local server is running, you can use:

```bash
curl http://localhost:11434
```

If you meant something else by “olama,” I don’t know.


In [5]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [6]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment registration can I join course after it started FAQ"}', call_id='call_OTFSyuHH6BWa2n6vuttaqXa0', name='search', type='function_call', id='fc_0626e04cd91f0a15006a294be02834819298bced1fcb44a342', namespace=None, status='completed')]

In [9]:
call=response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment registration can I join course after it started FAQ"}', call_id='call_OTFSyuHH6BWa2n6vuttaqXa0', name='search', type='function_call', id='fc_0626e04cd91f0a15006a294be02834819298bced1fcb44a342', namespace=None, status='completed')

In [11]:
import json
args=json.loads(call.arguments)
results=search(**args)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have r

In [19]:
results_json=json.dumps(results,indent=2)
print(results_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "977bf7786c",
    "course": "llm-zoomcamp",
    "section": "General Course

In [ ]:
function_call_output={
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}